<a href="https://colab.research.google.com/github/rkfussell/Physics_Affinity/blob/main/Confidence.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

import pandas as pd
import numpy as np
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
login()

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


confidence = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Confidence")
interest = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Interest & Use")
confidence_codebook = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Codebook-Confidence")
interest_codebook = pd.read_excel("/content/drive/My Drive/Colab Notebooks/2024_fall_Cornell_Phys1101and2207_physaff_Rnd2_CollectiveCoding.xlsx",sheet_name="Codebook-InterestandUse")

Mounted at /content/drive


In [ ]:
confidence.head()

,Unnamed: 0,course,studentID,In a few (short) sentences: How confident are you now in\nyour ability to learn physics? What contributes to this?,Confident,Somewhat Confident,Not Confident,Prior Coursework,No Prior Coursework,Time since last class,math confident,Not math confident,skill transfer,effort,inadequacy,covid,(other),Unnamed: 17,Unnamed: 18,Unnamed: 19
0,NaN,PHYS 2207,85ef67d0-649f-4e26-9545-1382bd4086a3,I have never taken a physics class in college ...,NaN,x,NaN,NaN,x,NaN,x,NaN,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,PHYS 2207,2d8769ee-10f4-4635-8a2e-f8e6af40abd3,I am pretty confident in my abilities to learn...,x,NaN,NaN,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,PHYS 2207,b6b37f0b-dad5-4d8c-acbe-4a72a693eb6e,Physics is pretty hard for me to grasp however...,NaN,NaN,NaN,NaN,NaN,NaN,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,PHYS 2207,f8f33e36-9c73-4007-a43b-3fcf1122f2ab,I am not confident in my ability to learn phys...,NaN,NaN,x,NaN,NaN,NaN,NaN,x,x,NaN,x,NaN,NaN,NaN,NaN,NaN
4,NaN,PHYS 2207,dbb358dd-c4e8-4714-84d9-a32e477edfd8,not confident. My AP Physics teacher in high s...,NaN,NaN,x,x,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:

confidence = confidence.rename(columns={"In a few (short) sentences: How confident are you now in\nyour ability to learn physics? What contributes to this?":"text"})
confidence = confidence.fillna(0)
confidence = confidence.replace('x',1)

/tmp/ipython-input-3397951919.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  confidence = confidence.replace('x',1)


In [ ]:
confidence

,Unnamed: 0,course,studentID,text,Confident,Somewhat Confident,Not Confident,Prior Coursework,No Prior Coursework,Time since last class,math confident,Not math confident,skill transfer,effort,inadequacy,covid,(other),Unnamed: 17,Unnamed: 18,Unnamed: 19
0,0.0,PHYS 2207,85ef67d0-649f-4e26-9545-1382bd4086a3,I have never taken a physics class in college ...,0,1,0,0,1,0,1,0,1,0,0,0,0,0,0,0
1,0.0,PHYS 2207,2d8769ee-10f4-4635-8a2e-f8e6af40abd3,I am pretty confident in my abilities to learn...,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2,0.0,PHYS 2207,b6b37f0b-dad5-4d8c-acbe-4a72a693eb6e,Physics is pretty hard for me to grasp however...,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0.0,PHYS 2207,f8f33e36-9c73-4007-a43b-3fcf1122f2ab,I am not confident in my ability to learn phys...,0,0,1,0,0,0,0,1,1,0,1,0,0,0,0,0
4,0.0,PHYS 2207,dbb358dd-c4e8-4714-84d9-a32e477edfd8,not confident. My AP Physics teacher in high s...,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
667,0.0,PHYS 2207,d919df7f-ae2e-486f-b563-8d7780fbc994,I don't feel super confident since I tend to a...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
668,0.0,PHYS 2207,b4e7cbfc-8d41-4e31-b841-89b8dbf1d0ba,I am somewhat confident because I took AP phys...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
669,0.0,PHYS 1101,35f501b0-2ac2-4666-bd69-aa71af516978,"Fairly. I believe if I put on enough time, I c...",0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
670,0.0,PHYS 1101,882ec822-d244-482f-9fea-b2abba2f1643,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
#load model
model_name = "google/gemma-3-4b-it"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    output_hidden_states=True,
    dtype=torch.float32,
    device_map="auto"
)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:

prompt = "You are applying a codebook that takes in a short response provided by a student in a physics class (<sent></sent>and assigns one of the following levels of confidence: <class>Level 0</class>, <class>Level 1</class>, or <class>Level 2</class>. \n"

prompt+= "<class>Level 0</class>, corresponds to 'not confident' and should be assigned when the student expresses only negative confidence ('not really confident', 'not confident', etc.) with no positive supportive descriptions about physics. This includes expressing concern, instead of confidence explicitly. "

prompt+= "Examples of <class>Level 0</class> confidence include: (a) <sent>'Not very confident because the last time I learned physics was freshman year of high school.'</sent> and (b) <sent>'I am slightly concerned because I did not do great in high school at physics. Also I think the majority students here are better at math than me.</sent> '\n"

prompt+= "<class>Level 1</class> corresponds to 'somewhat confident' and should be assigned when the student expresses both a reason to be confident, and a caveat about physics. If you're unsure; lean on the follow up statements instead of hedging language (even if they say 'somewhat'). "

prompt+= "Examples of <class>Level 1</class> confidence include: (a) <sent>'I have taken high school physics as well as an MCAT course this past summer which covered physics. This is made me feel somewhat confident in my ability to learn physics because I have been exposed to the topics. I do however feel that I will need to put more time into physics than I do in other classes because I know it is not my strong suit from past experiences.'</sent> and (b) <sent>'Right now, I am only mildly confident in my ability to learn physics. I have previous experience with the subject from high school, having taken the virtual AP course.'</sent> \n"

prompt+="<class>Level 2</class> corresponds to 'confident' and should be assigned when the student expresses only positive confidence, without caveats. This may include language such as 'reasonably', 'fairly', and 'at an 85%'. Hedging language is indistinguishable between humble and real feelings, so rely on the follow up statements to put this hedging language in context. "

prompt+="Examples of <class>Level 2</class> confidence include: (a) <sent>'I am reasonably confident in my ability to learn physics because I feel like I have a adequate background in math and other problem solving skills.'</sent> and (b) <sent>'I am fairly confident in my abilities to learn physics.  I have some prior knowledge in the subject and believe I can be successful in the class if I put in effort.  I also know myself to be a hard-worker and hope to seek help if I find myself needing it.'</sent> \n "

prompt+= "Classify this sentence: "













In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Run inference
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=3)

In [ ]:
# Decode the generated response
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("Model response:")
print(response)

Model response:
You are applying a codebook that takes in a short response provided by a student in a physics class and assigns one of the following levels of confidence: Level 0, Level 1, or Level 2.
 Level 0 corresponds to 'not confident' and should be assigned when the student expresses only negative confidence ('not really confident', 'not confident', etc.) with no positive supportive descriptions about physics. This includes expressing concern, instead of confidence explicitly. Examples of Level 0 confidence include: (a) 'Not very confident because the last time I learned physics was freshman year of high school.' and (b) 'I am slightly concerned because I did not do great in high school at physics. Also I think the majority students here are better at math than me. '
Level 1 corresponds to 'somewhat confident' and should be assigned when the student expresses both a reason to be confident, and a caveat about physics. If you're unsure; lean on the follow up statements instead of h

In [ ]:
confidence_high = confidence.loc[confidence["Confident"] == 1]
confidence_med = confidence.loc[confidence["Somewhat Confident"] == 1]
confidence_low = confidence.loc[confidence["Not Confident"] == 1]

In [ ]:
base=prompt

count_correct, count_wrong=0,0
for i in range(0,5):

    prompt=base+confidence_high.iloc[i]['text']
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # Run inference
    with torch.no_grad():
      outputs = model.generate(**inputs, max_new_tokens=100)
    # Decode the generated response
    ans = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    print("\n")
    print(i)
    print(ans)
    #if "Highly Confident" in ans:
    #    count_correct+=1
    #elif "Somewhat Confident" in ans:
    #    count_wrong+=1
    #    #print(confidence_high.iloc[i]['text'])
    #elif "Not Confident" in ans:
    #    count_wrong+=1
    #    #print(confidence_high.iloc[i]['text'])
    #else:
        #print(ans)
        #print(confidence_high.iloc[i]['text'])

print(count_correct)
print(count_wrong)






0
You are applying a codebook that takes in a short response provided by a student in a physics class (<sent></sent>and assigns one of the following levels of confidence: <class>Level 0</class>, <class>Level 1</class>, or <class>Level 2</class>. 
<class>Level 0</class>, corresponds to 'not confident' and should be assigned when the student expresses only negative confidence ('not really confident', 'not confident', etc.) with no positive supportive descriptions about physics. This includes expressing concern, instead of confidence explicitly. Examples of <class>Level 0</class> confidence include: (a) <sent>'Not very confident because the last time I learned physics was freshman year of high school.'</sent> and (b) <sent>'I am slightly concerned because I did not do great in high school at physics. Also I think the majority students here are better at math than me.</sent> '
<class>Level 1</class> corresponds to 'somewhat confident' and should be assigned when the student expresses bot

In [ ]:
from transformers import pipeline

In [ ]:
gpt2_pipeline = pipeline(task="text-generation", model = "openai-community/gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Device set to use cpu


In [ ]:
print(gpt2_pipeline("What if AI", max_new_tokens = 10, num_return_sequences = 2))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


[{'generated_text': 'What if AI is in control?\n\nYou can create your'}, {'generated_text': 'What if AI could do it?\n\nAI is just a'}]


In [ ]:
import os
from huggingface_hub import InferenceClient

client = InferenceClient(
    provider = 'together',
    api_key=os.environ["HF_token"]
)

KeyError: 'HF_token'

In [ ]:
completion = client.chat.completions.create(
    model = "deepseek-ai/DeepSeek-V3",
    messages = [
        {
            "role":"user",
            "content":"What is the capital of France?"
        }
    ]
)

In [ ]:
from datasets import load_dataset

wikipedia = load_dataset("wikimedia/wikipedia","20231101.en")

# Filter the documents
filtered = wikipedia.filter(lambda row: "football" in row["text"])

# Create a sample dataset
example = filtered.select(range(1))

print(example[0]["text"])

Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

20231101.en/train-00000-of-00041.parquet:   0%|          | 0.00/420M [00:00<?, ?B/s]

20231101.en/train-00001-of-00041.parquet:   0%|          | 0.00/351M [00:00<?, ?B/s]

20231101.en/train-00002-of-00041.parquet:   0%|          | 0.00/329M [00:00<?, ?B/s]

20231101.en/train-00003-of-00041.parquet:   0%|          | 0.00/331M [00:00<?, ?B/s]

...

20231101.en/train-00040-of-00041.parquet:   0%|          | 0.00/422M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6407814 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/41 [00:00<?, ?it/s]

Filter:   0%|          | 0/6407814 [00:00<?, ? examples/s]

AttributeError: 'DatasetDict' object has no attribute 'select'

In [ ]:
# Filter the documents
filtered = wikipedia.filter(lambda row: "football" in row["text"])

# Create a sample dataset
example = filtered.select(range(1))

print(example[0]["text"])